In [ ]:
!pip install optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.4/233.4 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# load datasets
diabetes = pd.read_csv('/content/drive/MyDrive/Bimbing/Hyperparameter Tuning/diabetes.csv')
heart = pd.read_csv('/content/drive/MyDrive/Bimbing/Hyperparameter Tuning/heart_failure.csv')
live = pd.read_csv('/content/drive/MyDrive/Bimbing/Hyperparameter Tuning/fb_live.csv')

In [ ]:
# the usual splitting
from sklearn.model_selection import train_test_split

X = heart.drop(columns='DEATH_EVENT').to_numpy()
y = heart[['DEATH_EVENT']].to_numpy()
y = y.reshape(len(y),)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    # Definisikan hiperparameter yang ingin dioptimalkan
    n_estimators = trial.suggest_int('n_estimators', 10, 100)
    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    # Buat model dengan hiperparameter yang dipilih
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, max_features=max_features)

    # Hitung skor cross-validation menggunakan hiperparameter yang dipilih
    scores = cross_val_score(model, X, y, cv=5, scoring='f1')

    # Kembalikan rata-rata skor cross-validation sebagai nilai objektif
    return scores.mean()

# Buat Optuna Study
study = optuna.create_study(direction='maximize')

# Jalankan proses optimasi
study.optimize(objective, n_trials=100)

# Dapatkan hasil terbaik
best_params = study.best_params
best_value = study.best_value
print(f"Best parameters: {best_params}")
print(f"Best value: {best_value}")

[I 2024-04-20 08:17:53,852] A new study created in memory with name: no-name-6d2a44cb-decf-4456-9726-3e4135d4dc06
[I 2024-04-20 08:17:54,649] Trial 0 finished with value: 0.40453920766889284 and parameters: {'n_estimators': 24, 'max_depth': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.40453920766889284.
[I 2024-04-20 08:17:55,899] Trial 1 finished with value: 0.47420365571050505 and parameters: {'n_estimators': 79, 'max_depth': 21, 'max_features': 'log2'}. Best is trial 1 with value: 0.47420365571050505.
[I 2024-04-20 08:17:56,994] Trial 2 finished with value: 0.44809424809424814 and parameters: {'n_estimators': 76, 'max_depth': 6, 'max_features': 'log2'}. Best is trial 1 with value: 0.47420365571050505.
[I 2024-04-20 08:17:57,625] Trial 3 finished with value: 0.5053301876973251 and parameters: {'n_estimators': 49, 'max_depth': 23, 'max_features': 'log2'}. Best is trial 3 with value: 0.5053301876973251.
[I 2024-04-20 08:17:58,313] Trial 4 finished with value: 0.46734771510

Best parameters: {'n_estimators': 16, 'max_depth': 3, 'max_features': 'sqrt'}
Best value: 0.5864332619064078


In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Objective function for Optuna optimization
def objective(trial):

    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 700),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy'])
    }


    # Create and train Random Forest classifier
    rf_classifier = RandomForestClassifier(**rf_params)
    rf_classifier.fit(X_train, y_train)

    # Calculate accuracy on validation set
    y_pred = rf_classifier.predict(X_test)
    accuracy = accuracy_score(y_pred, y_test)

    return accuracy

# Optimize parameters using Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Print best parameters and accuracy
best_params = study.best_params
best_accuracy = study.best_value
print("Best parameters:", best_params)
print("Best accuracy:", best_accuracy)

[I 2024-04-20 08:19:17,222] A new study created in memory with name: no-name-18ccfb9a-a706-4605-9539-ffefea70213d
[I 2024-04-20 08:19:18,292] Trial 0 finished with value: 0.7333333333333333 and parameters: {'n_estimators': 536, 'min_samples_split': 8, 'min_samples_leaf': 5, 'bootstrap': True, 'criterion': 'entropy'}. Best is trial 0 with value: 0.7333333333333333.
[I 2024-04-20 08:19:19,140] Trial 1 finished with value: 0.7333333333333333 and parameters: {'n_estimators': 536, 'min_samples_split': 9, 'min_samples_leaf': 4, 'bootstrap': True, 'criterion': 'gini'}. Best is trial 0 with value: 0.7333333333333333.
[I 2024-04-20 08:19:19,770] Trial 2 finished with value: 0.7333333333333333 and parameters: {'n_estimators': 442, 'min_samples_split': 9, 'min_samples_leaf': 3, 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.7333333333333333.
[I 2024-04-20 08:19:19,933] Trial 3 finished with value: 0.75 and parameters: {'n_estimators': 111, 'min_samples_split': 8, 'min_

Best parameters: {'n_estimators': 197, 'min_samples_split': 7, 'min_samples_leaf': 3, 'bootstrap': True, 'criterion': 'entropy'}
Best accuracy: 0.7666666666666667


In [ ]:
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# Split data

X_valid = x_shipping_test
y_valid = y_shipping_test

# Objective function for Optuna optimization
def objective(trial):
    # Define parameters to search
    tfidf_params = {
        'ngram_range': trial.suggest_categorical('ngram_range', [(1, 1), (1, 2), (1, 3)]),
        'max_df': trial.suggest_uniform('max_df', 0.25, 1.0),
        'min_df': trial.suggest_int('min_df', 1, 4)
    }

    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 700),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy'])
    }

    # Create TF-IDF vectorizer
    tfidf_vectorizer = TfidfVectorizer(**tfidf_params)
    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
    X_valid_tfidf = tfidf_vectorizer.transform(X_valid)

    # Create and train Random Forest classifier
    rf_classifier = RandomForestClassifier(**rf_params)
    rf_classifier.fit(X_train_tfidf, y_train)

    # Calculate accuracy on validation set
    y_pred = rf_classifier.predict(X_valid_tfidf)
    accuracy = accuracy_score(y_valid, y_pred)

    return accuracy

# Optimize parameters using Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Print best parameters and accuracy
best_params = study.best_params
best_accuracy = study.best_value
print("Best parameters:", best_params)
print("Best accuracy:", best_accuracy)

In [ ]:
style 1
X train -> scaling
X val -> scaling

style 2
X -> scaling
X train
X val

isolate
X test ????

In [ ]:
# X -> scaling, handling outlier -> train val

# X test -> data real -> pred vs actual ???

scaler = MinMax()
scaler.fit(X_train)
X_std = scaler.transform(X_train)

scaler menyiman min max dari data training


age
data train = min 10 tahun, max 70 tahun

X_std = scaler.transform(X_test)

data test -> si A umur 30 tahun
X - Xmax / (x_max - X_min)

train -> X1 X2 X3
test -> X1 X2 X3 X4 -> errro

test -> X1 X2 X3 -> pred vs actual -> ???

In [ ]:
prediksi F1 score 80%

FN dan FP -> error -> dampak error -> lost 1jt hari

FP 3
A : 100k -> lost 100k
B : 50k -> lost 50k
C : 200k -> lost 200k

TP dan TN -> benar -> gain. uplift 3jt

TP :
D -> 300k -> cancel -> waiting list -> E, akan lost 300k -> gagal lost
F
G
H

In [ ]:
# Presion: 98
# recall : 0.02



In [ ]:
prediksi suatu ads (fb, ig, dsb) yang akan berhasil

100rb -> gagal/berhasil (beli)

prediksi suatu ads akan gaga/berhasil -> gagal user akan stop, klo berhasil user akan perpanjang

FP : 98 bagus
FN : 0.2 jelek

FP : prediksi berhasil tp sebenarnya ads akan gagal : -> user sudah menghabiskan uang banyak tp gagal, uang user supaya tidak boncos
FN : prediksi gagal tp sebenarnya ads akan berhasil : -> user akan stop, uang yang masuk ke vendor akan sedikit

In [ ]:
# user churn dan user yang order di next month

# loyal dikit : 1000 -> 100 loyal : imbalance
# smote, tdk under karena data sedikit, class weight -> nothing
# bagaimana mengeluarkan pola loyal dan churn kepada model

# loyal -> outlier
# churn -> tdk outlier (mayoritas) -> outlier??

# loyal -> smote
# churn ->


# handling outlier tapi hanya pada kelas churn
# increase f1 score

# loyal : f1 score 38% 100 user -> improve -> outlier P95
# churn : f1 score churn 80% 900 user